# Rerun cross-fit SR-PDS with per-fold recovery

The original CF run stored only the union over folds (a term found in any fold
read as 1.0). This rerun records, per replication, the **fraction of folds**
that recovered each term (3 of 5 = 0.6) - the diagnostic that explains the
coverage. Same seeds and per-design budget, so coverage reproduces.

**Nothing existing is overwritten.** The original `cf_improved.pkl` is left
untouched; this writes to `cf_improved_foldrec.pkl` and `cf_fold_recovery.csv`.

Requires the updated `srpds_cf.py` and `cf_parallel.py`. ~2 hours on 4 cores
(dgp7 is the heavy one). Resumes by design if interrupted.


## Setup


In [ ]:
import os, time, pickle
import numpy as np, pandas as pd
import config as C
from dgp import DGP_REGISTRY
from cf_parallel import run_cf_parallel

N_JOBS_CF = 4          # 5 GB free -> 4 workers; raise if you free RAM
N_REPS_CF = 100

# NEW file -- the original cf_improved.pkl is never touched
CF_PKL = C.RESULTS_DIR / 'cf_improved_foldrec.pkl'
print('rerun writes to:', CF_PKL.name)
print('original cf_improved.pkl left as-is:',
      (C.RESULTS_DIR / 'cf_improved.pkl').exists())
print('results dir:', C.RESULTS_DIR.resolve())


## Rerun CF (severe first, per-fold recovery stored)

Heavy budget on `dgp7` only; cheap on `dgp6`/`dgp8`. Saves to the new file after
each design and skips designs already in it, so a restart resumes.


In [ ]:
CF_PER_DGP = {
    'dgp7': {'n_folds': 10, 'niterations': 100, 'parsimony_d': 0.0035},
    'dgp6': {'n_folds': 5,  'niterations': 40,  'parsimony_d': 0.01},
    'dgp8': {'n_folds': 5,  'niterations': 40,  'parsimony_d': 0.01},
}
DGP_ORDER = ['dgp7', 'dgp6', 'dgp8']

if CF_PKL.exists():
    prev = pd.read_pickle(CF_PKL); done = set(prev['dgp'].unique()); all_reps = [prev]
    print('resuming; done:', sorted(done))
else:
    done, all_reps = set(), []

print(f'CF on {N_JOBS_CF} cores, {N_REPS_CF} reps/design')
t0 = time.time()
for dgp_key in DGP_ORDER:
    if dgp_key in done:
        continue
    cfg = CF_PER_DGP[dgp_key]
    print(f'\n=== {dgp_key} ({DGP_REGISTRY[dgp_key]["label"]})  cfg={cfg} ===', flush=True)
    df = run_cf_parallel(DGP_REGISTRY[dgp_key]['fn'], n_reps=N_REPS_CF,
                         n=C.N_HEADLINE, p=C.P, s=C.S, beta0=C.BETA0,
                         cf_config=cfg, n_jobs=N_JOBS_CF, desc=dgp_key)
    df['dgp']=dgp_key; df['estimator']='sr_pds_cf'
    df['dgp_label']=DGP_REGISTRY[dgp_key]['label']; df['est_label']='SR-PDS (CF)'
    df['beta0']=C.BETA0
    all_reps.append(df)
    pd.concat(all_reps, ignore_index=True).to_pickle(CF_PKL)
    ok = df[~df['failed']]
    cov = ((ok['ci_low'] < C.BETA0) & (C.BETA0 < ok['ci_high'])).mean()
    print(f'  coverage={cov:.3f}  [{(time.time()-t0)/60:.1f} min]', flush=True)
cf_foldrec = pd.concat(all_reps, ignore_index=True)
print('\nCF done -> cf_improved_foldrec.pkl (with per-fold recovery)')


## Per-fold recovery table

Averages the per-fold recovery fraction across replications for each true term.
`mean_fold_pre` = mean fraction of folds that discovered the term; `mean_fold_post` = mean fraction that kept it. Writes `cf_fold_recovery.csv`.


In [ ]:
from evaluate import _term_label

cf = pd.read_pickle(C.RESULTS_DIR / 'cf_improved_foldrec.pkl')
rows = []
for dgp_key, g in cf.groupby('dgp'):
    e = DGP_REGISTRY[dgp_key]
    for eq, terms in (('y', e.get('truth_terms_y', [])), ('d', e.get('truth_terms_d', []))):
        pre_col, post_col = f'fold_pre_{eq}', f'fold_post_{eq}'
        if pre_col not in g.columns:
            continue
        for t in terms:
            key = str(t)
            pre  = np.mean([(row or {}).get(key, 0.0) for row in g[pre_col]])
            post = np.mean([(row or {}).get(key, 0.0) for row in g[post_col]])
            rows.append({'dgp': dgp_key, 'dgp_label': e['label'], 'equation': eq,
                         'term': key, 'term_label': _term_label(t),
                         'mean_fold_pre': round(float(pre), 3),
                         'mean_fold_post': round(float(post), 3),
                         'n_reps': len(g)})
cf_fold_recovery = pd.DataFrame(rows).sort_values(['dgp','equation','term_label'])
cf_fold_recovery.to_csv(C.RESULTS_DIR / 'cf_fold_recovery.csv', index=False)
print('wrote cf_fold_recovery.csv')
cf_fold_recovery


## Done

New files: `cf_improved_foldrec.pkl` (CF reps with per-fold recovery) and
`cf_fold_recovery.csv` (averaged per-fold rates). The original `cf_improved.pkl`
is unchanged - coverage is identical between the two, only the per-fold recovery
is added.
